# 12.7 — Graph pooling

Pooling decides what survives when node features become graph features.

Pooling follows message passing when labels belong to a whole graph or compressed supernodes. The score or assignment matrix defines which evidence remains visible.

Save a copy to Drive to edit.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

SEED = 12610
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def softmax(x):
    values = np.asarray(x, dtype=float)
    shifted = values - np.max(values)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum()


def leaky_relu(x, slope=0.2):
    values = np.asarray(x, dtype=float)
    return np.where(values >= 0.0, values, slope * values)


def graph_to_arrays(graph):
    nodes = list(graph.nodes())
    adjacency = nx.to_numpy_array(graph, nodelist=nodes, dtype=float)
    labels = np.array([graph.nodes[node].get("label", 0) for node in nodes], dtype=int)
    features = np.array([graph.nodes[node].get("x", [0.0, 0.0, 0.0]) for node in nodes], dtype=float)
    return nodes, adjacency, features, labels


def add_features(graph, labels, noise=0.05, seed=0):
    rng = np.random.default_rng(seed)
    degrees = np.array([degree for _, degree in graph.degree()], dtype=float)
    degree_scale = max(float(degrees.max()), 1.0)
    for index, node in enumerate(graph.nodes()):
        label = int(labels[index])
        base = np.array([1.0 - label, label, degrees[index] / degree_scale], dtype=float)
        graph.nodes[node]["label"] = label
        graph.nodes[node]["x"] = base + rng.normal(0.0, noise, size=3)
    return graph


def make_toy_graph():
    graph = nx.Graph()
    graph.add_edges_from([(0, 1), (1, 2), (2, 3), (0, 2)])
    labels = np.array([0, 0, 1, 1], dtype=int)
    return add_features(graph, labels, noise=0.0, seed=1)


def make_karate_graph():
    graph = nx.karate_club_graph()
    labels = np.array([0 if graph.nodes[node]["club"] == "Mr. Hi" else 1 for node in graph.nodes()], dtype=int)
    return add_features(graph, labels, noise=0.03, seed=2)


def make_sbm_graph(sizes, p_in, p_out, noise, seed):
    probs = [[p_in, p_out], [p_out, p_in]]
    graph = nx.stochastic_block_model(sizes, probs, seed=seed)
    labels = []
    for block, size in enumerate(sizes):
        labels.extend([block] * size)
    return add_features(graph, np.array(labels, dtype=int), noise=noise, seed=seed + 100)


def build_graph_ladder():
    return [
        {"name": "D1 toy 4-node", "graph": make_toy_graph(), "complexity": 1},
        {"name": "D2 karate club", "graph": make_karate_graph(), "complexity": 2},
        {"name": "D3 clean SBM", "graph": make_sbm_graph([18, 18], 0.34, 0.04, 0.07, 3), "complexity": 3},
        {"name": "D4 larger SBM", "graph": make_sbm_graph([35, 35], 0.20, 0.025, 0.10, 4), "complexity": 4},
        {"name": "D5 noisy overlap", "graph": make_sbm_graph([45, 45], 0.16, 0.075, 0.22, 5), "complexity": 5},
    ]


def class_train_mask(labels):
    mask = np.zeros(len(labels), dtype=bool)
    for label in np.unique(labels):
        candidates = np.where(labels == label)[0]
        count = max(1, len(candidates) // 4)
        mask[candidates[:count]] = True
    return mask


def centroid_predict(embeddings, labels, train_mask):
    classes = np.unique(labels)
    centroids = []
    for label in classes:
        rows = embeddings[(labels == label) & train_mask]
        centroids.append(rows.mean(axis=0))
    centroids = np.vstack(centroids)
    distances = ((embeddings[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)
    return classes[np.argmin(distances, axis=1)]


def aligned_accuracy(predictions, labels):
    raw = float(np.mean(predictions == labels))
    flipped = float(np.mean(1 - predictions == labels))
    return max(raw, flipped)


def pairwise_auc(positive_scores, negative_scores):
    total = 0.0
    count = 0.0
    for positive in positive_scores:
        for negative in negative_scores:
            if positive > negative:
                total += 1.0
            elif positive == negative:
                total += 0.5
            count += 1.0
    return total / count


def sample_link_split(graph, seed=0, holdout_fraction=0.2):
    rng = np.random.default_rng(seed)
    edges = [tuple(sorted(edge)) for edge in graph.edges()]
    rng.shuffle(edges)
    holdout_count = max(1, int(len(edges) * holdout_fraction))
    positive_edges = edges[:holdout_count]
    train_graph = graph.copy()
    train_graph.remove_edges_from(positive_edges)
    non_edges = [tuple(sorted(edge)) for edge in nx.non_edges(graph)]
    rng.shuffle(non_edges)
    negative_edges = non_edges[:holdout_count]
    return train_graph, positive_edges, negative_edges


def spectral_embeddings(adjacency, dimensions=8):
    adjacency = np.asarray(adjacency, dtype=float)
    augmented = adjacency + np.eye(adjacency.shape[0])
    degrees = augmented.sum(axis=1)
    inv_sqrt = 1.0 / np.sqrt(np.maximum(degrees, 1e-12))
    normalized = inv_sqrt[:, None] * augmented * inv_sqrt[None, :]
    values, vectors = np.linalg.eigh(normalized)
    order = np.argsort(values)[::-1]
    keep = order[: min(dimensions, adjacency.shape[0])]
    return vectors[:, keep] * values[keep]


def plot_graph_panels(results, metric_name):
    fig, axes = plt.subplots(1, len(results), figsize=(15, 3))
    for axis, result in zip(axes, results):
        graph = result["graph"]
        colors = result.get("colors")
        pos = nx.spring_layout(graph, seed=SEED)
        nx.draw_networkx_nodes(graph, pos, node_color=colors, cmap="coolwarm", node_size=55, ax=axis)
        nx.draw_networkx_edges(graph, pos, alpha=result.get("edge_alpha", 0.25), width=0.8, ax=axis)
        axis.set_title(f"{result['name']}\n{metric_name}={result['metric']:.3f}")
        axis.axis("off")
    plt.tight_layout()
    plt.show()
    plt.figure(figsize=(5, 3))
    plt.plot([result["complexity"] for result in results], [result["metric"] for result in results], marker="o")
    plt.xticks([1, 2, 3, 4, 5], ["D1", "D2", "D3", "D4", "D5"])
    plt.ylabel(metric_name)
    plt.xlabel("graph ladder rung")
    plt.grid(True, alpha=0.3)
    plt.show()

def pool_graph(adjacency, features, assignment, scores, k=2):
    assignment = np.asarray(assignment, dtype=float)
    sum_pool = features.sum(axis=0)
    mean_pool = features.mean(axis=0)
    assigned_features = assignment.T @ features
    assigned_adjacency = assignment.T @ adjacency @ assignment
    top_nodes = np.argsort(scores)[-k:][::-1]
    return {
        "sum": sum_pool,
        "mean": mean_pool,
        "assigned_features": assigned_features,
        "assigned_adjacency": assigned_adjacency,
        "top_nodes": top_nodes,
    }


def pooled_node_predictions(graph):
    nodes, adjacency, features, labels = graph_to_arrays(graph)
    communities = list(nx.algorithms.community.greedy_modularity_communities(graph))
    assignment = np.zeros((len(nodes), len(communities)))
    node_to_index = {node: index for index, node in enumerate(nodes)}
    for community_index, community in enumerate(communities):
        for node in community:
            assignment[node_to_index[node], community_index] = 1.0
    pooled = assignment.T @ features
    train_mask = class_train_mask(labels)
    predictions = np.zeros_like(labels)
    for community_index in range(assignment.shape[1]):
        members = np.where(assignment[:, community_index] > 0.0)[0]
        train_members = [member for member in members if train_mask[member]]
        if len(train_members) == 0:
            nearest = int(np.argmin(((features[members].mean(axis=0) - pooled) ** 2).sum(axis=1)))
            predictions[members] = nearest % 2
        else:
            votes = labels[train_members]
            predictions[members] = int(np.round(votes.mean()))
    return predictions, assignment, pooled, labels

## The concept, built once: readout, assignment, and top-k

Readout can be $r=\sum_i h_i$ or $r=\frac{1}{n}\sum_i h_i$. Assignment pooling uses $X'=S^\top X$ and $A'=S^\top A S$.

In [ ]:
features = np.array([[1.0, 0.0], [0.0, 1.0], [2.0, 1.0], [1.0, 1.0]])
adjacency = np.array([[0.0, 1.0, 1.0, 0.0], [1.0, 0.0, 0.0, 1.0], [1.0, 0.0, 0.0, 1.0], [0.0, 1.0, 1.0, 0.0]])
assignment = np.array([[1.0, 0.0], [1.0, 0.0], [0.0, 1.0], [0.0, 1.0]])
scores = np.array([0.2, 0.9, 0.1, 0.8])
pooled = pool_graph(adjacency, features, assignment, scores)
print("sum", pooled["sum"])
print("mean", pooled["mean"])
print("assigned features")
print(pooled["assigned_features"])
assert np.allclose(pooled["sum"], [4.0, 3.0])
assert np.allclose(np.round(pooled["mean"], 3), [1.000, 0.750])
assert np.allclose(pooled["assigned_features"], [[1.0, 1.0], [3.0, 2.0]])

Top-k pooling keeps high-scoring nodes, while assignment pooling also rewires the graph. The lesson values keep nodes $1$ and $3$ and produce $S^\top A S=\begin{bmatrix}2&2\\2&2\end{bmatrix}$.

In [ ]:
print("top-k nodes", pooled["top_nodes"].tolist())
print("pooled adjacency")
print(pooled["assigned_adjacency"])
assert pooled["top_nodes"].tolist() == [1, 3]
assert np.allclose(pooled["assigned_adjacency"], [[2.0, 2.0], [2.0, 2.0]])

## The dataset ladder

Family F11 uses one inline graph ladder for every topic: D1 is a 4-node toy graph, D2 is karate club, D3 is a stochastic block model, D4 is larger, and D5 is noisy/overlapping. The metric for this notebook is pooled node accuracy.

In [ ]:
ladder = build_graph_ladder()
for rung in ladder:
    graph = rung["graph"]
    nodes, adjacency, features, labels = graph_to_arrays(graph)
    counts = np.bincount(labels)
    print(rung["name"], "nodes", graph.number_of_nodes(), "edges", graph.number_of_edges(), "features", features.shape, "classes", counts.tolist())
    print("sample labels", labels[:8].tolist())

## Run the same pooling readout across D1-D5

In [ ]:
results = []
for rung in ladder:
    graph = rung["graph"]
    predictions, assignment, pooled, labels = pooled_node_predictions(graph)
    metric = aligned_accuracy(predictions, labels)
    retained = assignment.argmax(axis=1)
    results.append({"name": rung["name"], "complexity": rung["complexity"], "graph": graph, "metric": metric, "colors": retained})
    print(f"{rung['name']:<18} pooled-node-acc={metric:.3f} supernodes={assignment.shape[1]}")

## Results visualization

In [ ]:
plot_graph_panels(results, "pooled-acc")

## Pitfall on D5: hard top-k can drop a bridge

A bridge may have modest local feature score but carry the only route between regions. Pure top-k can remove it; bridge-aware scoring adds centrality.

In [ ]:
d5 = ladder[-1]["graph"]
centrality = nx.betweenness_centrality(d5)
nodes = list(d5.nodes())
feature_scores = np.array([d5.nodes[node]["x"][1] for node in nodes])
bridge_scores = np.array([centrality[node] for node in nodes])
k = max(4, len(nodes) // 8)
plain_keep = set(np.argsort(feature_scores)[-k:])
fixed_keep = set(np.argsort(feature_scores + 2.0 * bridge_scores)[-k:])
top_bridge = int(np.argmax(bridge_scores))
print("highest-bridge node kept by plain top-k", top_bridge in plain_keep)
print("highest-bridge node kept by bridge-aware top-k", top_bridge in fixed_keep)

## Evaluate it + Practice

- Metric: pooled node accuracy; compare it with a no-skill baseline such as majority class or random pair ranking.
- Sanity check: D1 should match the lesson arithmetic exactly before you trust D2-D5.
- Ablation: replace assignment pooling with mean-only pooling or top-k without bridge scores
- Failure signals: bridge nodes disappear, mean readouts for different-size graphs match, or supernode labels become mixed
- Baseline: assign every node to the largest pooled supernode

Practice prompts:
1. Change one graph-ladder noise value and predict which rung should move most.

2. Add a second diagnostic printout that exposes one intermediate tensor for D3.

3. Replace the simple readout with another CPU-only NumPy readout and compare the metric curve.